Alur ekstraksi fitur:
  1. Pre-Emphasis
  2. Frame Blocking
  3. Windowing (Hamming)
  4. Fast Fourier Transform (FFT)
  5. Mel Filter Bank
  6. Logarithm
  7. Discrete Cosine Transform (DCT)
  8. Deltas and Double Deltas

Formula:
  - Pre-Emphasis  : X(n) = y(n) - a × y(n-1)              [2.2]
  - Hamming Window: w(n) = 0.54 - 0.46 × cos(2πn / (N-1)) [2.3]
  - Mel Scale     : Mel(f) = 1127 × ln(1 + f/700)         [2.4]
  - DCT (MFCC)    : Cn = Σ log(sk) × cos(πn/K × (k-0.5))  [2.5]
  - Delta         : δ(t) = c(t+1) - c(t)                  [2.6]

In [ ]:
import numpy as np
import librosa
import soundfile as sf
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
sampled_root = '../data/sampled'
sampled2_root = '../data/sampled2'

In [ ]:
# ─────────────────────────────────────────────
#  Step 1 : Pre-Emphasis
# ─────────────────────────────────────────────
def pre_emphasis(signal : np.ndarray, a = 0.97):
    return np.append(signal[0],signal[1:] - (a* signal[:-1]))

In [ ]:
# ─────────────────────────────────────────────
#  Step 2 : Frame Blocking
# ─────────────────────────────────────────────
def frame_blocking(signal : np.ndarray, sample_rate : int,
                   frame_size, frame_distance ): 
    frame_length =(int)(round(frame_size * sample_rate))
    frame_diff   =(int)(round(frame_distance * sample_rate))
    signal_length = len(signal)
    n_frames = 1 + (signal_length - frame_length) // frame_diff 
    
    indices = np.array([
        signal[i * frame_diff : i * frame_diff + frame_length]
        for i in range(n_frames)
    ])
    
    indices = np.minimum (indices, signal_length - 1 )
    frames  = signal[indices]
    return frames

In [ ]:
# ─────────────────────────────────────────────
#  Step 3 : Windowing (Hamming)
# ─────────────────────────────────────────────
def windowing(frames : np.ndarray) :
    frame_Shape = frames.shapes[1]
    n = np.arange(frame_Shape)
    
    hamming = 0.54 - 0.46 * np.cos * (2 * np.pi * n // frame_Shape - 1)
    windowing  = frames * hamming
    return windowing

In [ ]:
# ─────────────────────────────────────────────
#  Step 4 : Fast Fourier Transform (FFT)
# ─────────────────────────────────────────────
def apply_fft(windowed_frames: np.ndarray):
    frame_length = windowed_frames.shape[1]        # ambil panjang 1 bingkai
    n_fft = 1 << (frame_length - 1).bit_length()  # cari perpangkatan 2 terdekat
    magnitude_spectrum = np.abs(np.fft.rfft(windowed_frames, n=n_fft))  # hitung FFT
    power_spectrum = (1.0 / n_fft) * (magnitude_spectrum ** 2)          # hitung daya
    return power_spectrum

In [ ]:
# ─────────────────────────────────────────────
#  Step 5 : Mel Filter Bank
# ─────────────────────────────────────────────
def mel_filter_bank(power_spectrum: np.ndarray,
                    sample_rate: int,
                    n_mels: int = 40,
                    fmin: float = 0.0,
                    fmax: float = None) -> np.ndarray:

    n_fft = (power_spectrum.shape[1] - 1) * 2   # turunkan n_fft dari power_spectrum
    fmax  = fmax or sample_rate / 2.0

    to_mel = lambda f: 1127.0 * np.log(1.0 + f / 700.0)
    to_hz  = lambda m: 700.0 * (np.exp(m / 1127.0) - 1.0)

    # Buat n_mels+2 titik merata di skala Mel lalu balik ke Hz
    mel_points = np.linspace(to_mel(fmin), to_mel(fmax), n_mels + 2)
    hz_points  = to_hz(mel_points)

    # Konversi ke indeks bin FFT
    bins = np.floor((n_fft + 1) * hz_points / sample_rate).astype(int)

    # Bangun matriks filter segitiga (n_mels × n_freq_bins)
    n_freq_bins   = n_fft // 2 + 1
    filter_matrix = np.zeros((n_mels, n_freq_bins))
    for m in range(1, n_mels + 1):
        l, c, r = bins[m - 1], bins[m], bins[m + 1]         # left, center, right
        if c != l: filter_matrix[m-1, l:c] = (np.arange(l, c) - l) / (c - l)  # sisi kiri
        if r != c: filter_matrix[m-1, c:r] = (r - np.arange(c, r)) / (r - c)  # sisi kanan

    # Terapkan filter → hindari log(0)
    return np.maximum(np.dot(power_spectrum, filter_matrix.T), 1e-10)

In [ ]:
# ─────────────────────────────────────────────
#  Step 6 : Logarithm
# ─────────────────────────────────────────────
def logarithm_log(filter_bank_energies: np.ndarray):
    return np.log(filter_bank_energies)

In [ ]:
# ─────────────────────────────────────────────
#  Step 7 : DCT
# ─────────────────────────────────────────────
def dct(log_energies: np.ndarray, n_mfcc: int = 13):
    K = log_energies.shape[1]                    # total filter Mel
    n = np.arange(n_mfcc)                        # indeks koefisien 0..n_mfcc-1
    k = np.arange(1, K + 1)                      # indeks filter 1..K
    
    phase = np.pi / K * (k - 0.5)               # hitung bagian (π/K * (k-0.5)) per filter
    cos_matrix = np.cos(np.outer(n, phase))      # outer: broadcast n × phase → (n_mfcc × K)

    return np.dot(log_energies, cos_matrix.T)    # (num_frames × K) dot (K × n_mfcc)

In [ ]:
# ─────────────────────────────────────────────
#  Step 8 : Deltas and Double Deltas
# ─────────────────────────────────────────────
def compute_deltas(mfcc: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    # Delta: selisih bingkai berurutan
    delta        = np.diff(mfcc,        axis=0, prepend=mfcc[:1])
    # Double delta: selisih dari delta
    double_delta = np.diff(delta,       axis=0, prepend=delta[:1])
    return delta, double_delta